# Paper 3 — Sensitivity (vii) rerun: Kaeng Khoi + Tan Diao combined

**Why:** Kaeng Khoi Hospital (facility 10807) was coded to Kaeng Khoi tambon (190201) until Dec 2024 and to Tan Diao (190203) from Jan 2025. In the extended series (Jan 2022 – Sep 2026) this moves visits between two tambons in the same month that the PM2.5 source changes, so the two tambons are merged into one unit here (Sect. 2.11 (vii)).

**Input:** `analysis_panel_total_lag0_satscale.csv` — the tambon × month panel written by notebook 20 for the extended satellite-scale run (5,533 rows, 109 tambons, 51 months).

**Model:** same as the sensitivity analyses in notebook 20 — negative binomial, tambon fixed effects, month-of-year + fiscal-year effects, offset log(population); exposure = within-tambon PM2.5 deviation per 10 µg/m³ (lag 0).

**Steps:**
1. Refit the panel *unchanged* and check that the total-visit IRR reproduces notebook 20 (1.003, 0.965–1.042).
2. Merge 190201 + 190203 (sum counts and population; population-weighted PM2.5), recompute the within-tambon deviation, refit.
3. Report total, respiratory, cardiovascular, eye and skin IRRs before and after merging → `sens_vii_merged_KK_TD.csv`.

Run: Runtime ▸ Run all (CPU is enough, ~1–2 min).

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
!pip install -q gdown statsmodels

In [ ]:
import os, io
PANEL = 'analysis_panel_total_lag0_satscale.csv'
PANEL_ID = '1A7BDqRLWdEsa419RB8QNwLBpJH6oBiBQ'
def fetch_from_drive(file_id, out):
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        auth.authenticate_user()
        req = build('drive', 'v3').files().get_media(fileId=file_id, supportsAllDrives=True)
        with io.FileIO(out, 'wb') as fh:
            dl, done = MediaIoBaseDownload(fh, req), False
            while not done: _, done = dl.next_chunk()
        print('Downloaded (Drive API):', out); return
    except Exception as ex:
        print('Drive API failed, trying gdown:', str(ex)[:100])
    import gdown; gdown.download(id=file_id, output=out, quiet=False)
if repo_file(PANEL): PANEL = repo_file(PANEL); print('Repository data:', PANEL)
elif not os.path.exists(PANEL): fetch_from_drive(PANEL_ID, PANEL)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv(PANEL, encoding='utf-8-sig', dtype={'a_code': str})
print(df.shape, '| tambons:', df.a_code.nunique(), '| months:', df.ym.nunique())

KK, TD = '190201', '190203'          # Kaeng Khoi, Tan Diao (Kaeng Khoi district)
for c in (KK, TD):
    s = df[df.a_code == c]
    print(c, s.name.iloc[0] if len(s) else '—', '| rows:', len(s), '| months with y>0:', int((s.y > 0).sum()),
          '| last month with y>0:', s.loc[s.y > 0, 'ym'].max() if (s.y > 0).any() else '—')

GROUPS = {
    'total':          ['COPD', 'Asthma', 'Pneumonia', 'Influenza', 'Acute pharyngitis', 'Chronic rhinitis', 'Bronchitis',
                       'Ischemic heart diseases', 'Cerebrovascular diseases (Stroke)', 'กลุ่มโรคตาอักเสบ', 'กลุ่มโรคผิวหนังอักเสบ'],
    'respiratory':    ['COPD', 'Asthma', 'Pneumonia', 'Influenza', 'Acute pharyngitis', 'Chronic rhinitis', 'Bronchitis'],
    'cardiovascular': ['Ischemic heart diseases', 'Cerebrovascular diseases (Stroke)'],
    'eye':            ['กลุ่มโรคตาอักเสบ'],
    'skin':           ['กลุ่มโรคผิวหนังอักเสบ'],
}
COUNTS = GROUPS['total']
missing = [c for c in COUNTS if c not in df.columns]
assert not missing, f'columns not found: {missing}'
assert np.allclose(df[COUNTS].sum(axis=1), df.y), 'y is not the sum of the 11 conditions — check the panel'

# check the exposure definition used by notebook 20: pm_w = (pm_use - tambon mean) / 10
chk = (df.pm_use - df.groupby('a_code').pm_use.transform('mean')) / 10
print('pm_w = (pm_use − tambon mean)/10 ? max abs diff =', float((chk - df.pm_w).abs().max()))

In [ ]:
def fit(d, outcome_cols, label):
    d = d.copy()
    d['yy'] = d[outcome_cols].sum(axis=1)
    m = smf.negativebinomial('yy ~ pm_w + C(a_code) + C(moy) + C(fy)', data=d, offset=d['log_pop'])
    r = m.fit(method='bfgs', maxiter=3000, disp=0)
    if not r.mle_retvals.get('converged', True):
        r = m.fit(method='newton', maxiter=200, disp=0, start_params=r.params)
    b, se = r.params['pm_w'], r.bse['pm_w']
    return {'scenario': label, 'IRR': np.exp(b), 'lo': np.exp(b - 1.96 * se), 'hi': np.exp(b + 1.96 * se),
            'p': r.pvalues['pm_w'], 'alpha': r.params['alpha'], 'n_rows': len(d), 'n_tambon': d.a_code.nunique(),
            'n_month': d.ym.nunique(), 'converged': r.mle_retvals.get('converged')}

# ---- step 1: reproduce notebook 20 on the unchanged panel ----
base = fit(df, GROUPS['total'], 'unchanged (109 tambons)')
print('unchanged total IRR = {IRR:.3f} ({lo:.3f}–{hi:.3f}); notebook 20: 1.003 (0.965–1.042)'.format(**base))
if abs(base['IRR'] - 1.0033) < 0.005:
    print('✓ reproduces notebook 20')
else:
    print('⚠ differs from notebook 20 — compare the before/after columns below, not with the old number')

In [ ]:
# ---- step 2: merge Kaeng Khoi + Tan Diao ----
pair = df[df.a_code.isin([KK, TD])].copy()
pop_fixed = pair.groupby('a_code').pop.first()                 # tambon population is constant over time
wide = pair.pivot_table(index='ym', columns='a_code', values=COUNTS + ['pm_use'], aggfunc='first')

months = sorted(pair.ym.unique())
rows = []
for ym in months:
    rec = {'a_code': '190201+190203', 'name': 'Kaeng Khoi + Tan Diao', 'ym': ym}
    for c in COUNTS:
        rec[c] = sum(wide[(c, t)].get(ym, 0) if (c, t) in wide.columns and pd.notna(wide[(c, t)].get(ym)) else 0 for t in (KK, TD))
    pm_kk = wide[('pm_use', KK)].get(ym) if ('pm_use', KK) in wide.columns else np.nan
    pm_td = wide[('pm_use', TD)].get(ym) if ('pm_use', TD) in wide.columns else np.nan
    w = {KK: pop_fixed.get(KK, 0), TD: pop_fixed.get(TD, 0)}
    if pd.notna(pm_kk) and pd.notna(pm_td):
        rec['pm_use'] = (pm_kk * w[KK] + pm_td * w[TD]) / (w[KK] + w[TD])
    else:                                                       # one tambon absent that month: use the other
        rec['pm_use'] = pm_td if pd.notna(pm_td) else pm_kk
        rec['pm_filled'] = True
    rows.append(rec)
merged = pd.DataFrame(rows)
merged['pop'] = pop_fixed.sum()
merged['y'] = merged[COUNTS].sum(axis=1)
meta = pair.drop_duplicates('ym').set_index('ym')[['moy', 'fy']]
merged = merged.join(meta, on='ym')
print('merged unit months:', len(merged), '| months with PM2.5 from one tambon only:', int(merged.get('pm_filled', pd.Series(dtype=bool)).fillna(False).sum()))

df2 = pd.concat([df[~df.a_code.isin([KK, TD])], merged], ignore_index=True)
df2['log_pop'] = np.log(df2['pop'])
df2['pm_w'] = (df2.pm_use - df2.groupby('a_code').pm_use.transform('mean')) / 10
print('panel after merging:', df2.shape, '| units:', df2.a_code.nunique())
assert df2.a_code.nunique() == df.a_code.nunique() - 1
assert int(df2[COUNTS].sum().sum()) == int(df[COUNTS].sum().sum()), 'visit total changed — check merge'

In [ ]:
# ---- step 3: refit all outcome groups, before and after merging ----
out = []
for g, cols in GROUPS.items():
    a = fit(df, cols, 'separate'); a['group'] = g
    b = fit(df2, cols, 'KK+TD merged'); b['group'] = g
    out += [a, b]
res = pd.DataFrame(out)[['group', 'scenario', 'IRR', 'lo', 'hi', 'p', 'alpha', 'n_rows', 'n_tambon', 'n_month', 'converged']]
res['IRR (95% CI)'] = res.apply(lambda r: f"{r.IRR:.2f} ({r.lo:.2f}–{r.hi:.2f})", axis=1)
res.to_csv('sens_vii_merged_KK_TD.csv', index=False, encoding='utf-8-sig')
pd.set_option('display.width', 160)
print(res[['group', 'scenario', 'IRR (95% CI)', 'p', 'n_tambon', 'n_rows', 'converged']].to_string(index=False))

resp = res[(res.group == 'respiratory') & (res.scenario == 'KK+TD merged')].iloc[0]
print('\nFor Sect. 2.11 / 3.7 (respiratory, lag 0, KK+TD merged): '
      f"IRR {resp.IRR:.2f} ({resp.lo:.2f}–{resp.hi:.2f}) per 10 µg/m³, {int(resp.n_tambon)} units, {int(resp.n_rows)} unit-months")

In [ ]:
try:
    from google.colab import files; files.download('sens_vii_merged_KK_TD.csv')
except ImportError:
    pass